In [4]:
import numpy as np
import json
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [6]:
# Chargement des données et vocab sauvegardés
Path_artifact = Path("C:/Users/223114186/Desktop/TP3_Word Embeddings/artifact")

data = np.load(Path_artifact / "text8_dataset.npz", allow_pickle=True)
X_ctx = data["X_ctx"]
y_tgt = data["y_tgt"]
unk_id = int(data["unk_id"])

with open(Path_artifact / "vocab_mappings.json", "r", encoding="utf-8") as f:
    voc_data = json.load(f)
    vocab = voc_data["vocab"]
    word2id = {k: int(v) for k, v in voc_data["word2id"].items()}
    id2word = {int(k): v for k, v in voc_data["id2word"].items()}

print("✅ Données et vocab rechargés :")
print("X_ctx:", X_ctx.shape, " y_tgt:", y_tgt.shape)
print("Vocab size:", len(vocab))

✅ Données et vocab rechargés :
X_ctx: (2424441, 4)  y_tgt: (2424441,)
Vocab size: 31167


In [10]:
vocab_size = len(vocab) + 1 # +UNK
embed_dim = 100


In [13]:
# Taille de la fenêtre
window = 2   # donc contexte total = 4 mots

# Construction du modèle CBOW
inputs = keras.Input(shape=(2 * window,), dtype='int32')  # ⬅️ ici
# Embedding: (batch, 2*window, embed_dim)
emb = layers.Embedding(vocab_size, embed_dim, name='emb')(inputs)

# ⚠️ Remplacer tf.reduce_mean(...) par une couche Keras :
mean = layers.GlobalAveragePooling1D()(emb)   # moyenne sur l’axe 'temps' (2*window)

outputs = layers.Dense(vocab_size, activation='softmax')(mean)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['sparse_categorical_accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ emb (Embedding)                 │ (None, 4, 100)         │     3,116,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 100)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 31168)          │     3,147,968 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,264,768 (23.90 MB)

 Trainable params: 6,264,768 (23.90 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
batch_size = 1024
ds = tf.data.Dataset.from_tensor_slices((X_ctx,
y_tgt)).shuffle(200_000).batch(batch_size).prefetch(2)
history = model.fit(ds, epochs=3)

Epoch 1/3
2368/2368 ━━━━━━━━━━━━━━━━━━━━ 613s 258ms/step - loss: 8.2150 - sparse_categorical_accuracy: 0.0884
Epoch 2/3
2368/2368 ━━━━━━━━━━━━━━━━━━━━ 610s 257ms/step - loss: 7.4768 - sparse_categorical_accuracy: 0.1055
Epoch 3/3
2368/2368 ━━━━━━━━━━━━━━━━━━━━ 671s 283ms/step - loss: 7.1926 - sparse_categorical_accuracy: 0.1104


In [15]:
# Récupération de la matrice d'embedding
E = model.get_layer('emb').get_weights()[0] # (vocab_size, embed_dim)
# Normalisation L2
E_norm = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
7
from numpy.linalg import norm
def topk_similar(word, k=10):
    wid = word2id.get(word, None)
    if wid is None:
        return []
    v = E_norm[wid]
    sims = E_norm @ v
    idx = np.argsort(-sims)[:k+1]
    words = [(id2word[i], float(sims[i])) for i in idx if i != wid]
    return words[:k]

print(topk_similar('man', k=10))


[('woman', 0.9153912663459778), ('angry', 0.8570120334625244), ('love', 0.8446749448776245), ('heaven', 0.8248910903930664), ('tell', 0.8227998614311218), ('evil', 0.8224375247955322), ('hears', 0.8149232864379883), ('saying', 0.8059643507003784), ('pregnant', 0.8043571710586548), ('glory', 0.8040385842323303)]
